# A/B Testing and Causal Inference — the Data-Scientist Differentiator

ML rounds test modeling; DS rounds test **experiment design and causal thinking**. These questions ("was the lift real?", "why not just compare converters vs non-converters?") are where product-DS loops are decided. Everything below is executed: power analysis before the test, peeking damage quantified, and Simpson's reversal on treatment effects.

## Q1 — The A/B Test Lifecycle, Stated as a Protocol

1. **Hypothesis & metric:** *one* primary metric (conversion), guardrail metrics (latency, revenue-per-user). No metric = no test.
2. **MDE and power analysis:** decide the minimum detectable effect *before* looking at data; compute n for α=0.05, power 0.8.
3. **Randomize and check balance:** unit of randomization = unit of analysis (user, not session — sessions of one user share outcomes).
4. **Run to the planned horizon.** No peeking (Q3).
5. **Analyze:** z-test/t-test on the primary metric; CUPED or regression adjustment for variance reduction; segment *exploratory* only.
6. **Decide + document:** ship/kill/iterate; log the result either way — the experiment repo is compounding institutional knowledge.

## Q2 — Power Analysis, Executed Before the Test

The question: *"we expect a 1-percentage-point lift on a 5% baseline — how many users?"* The formula families: for proportions, n ≈ 16·p(1−p)/δ² (the "16" comes from α=0.05, power=0.8). Running it is 30 seconds and prevents the two classic failures: underpowered tests that only detect huge effects, and post-hoc "the effect must be there" rationalization.

In [1]:
import numpy as np
from scipy import stats
import statsmodels.stats.api as sms

p0 = 0.05                    # baseline conversion
for lift_pp in (0.5, 1.0, 2.0):     # absolute lift in percentage points
    p1 = p0 + lift_pp / 100
    es = sms.proportion_effectsize(p0, p1)
    n = sms.NormalIndPower().solve_power(es, power=0.8, alpha=0.05)
    print(f"detect {lift_pp:.1f}pp on 5% base: n = {n:,.0f} per group")

# the sqrt relationship: halve the detectable effect -> 4x the sample
print("\nrule: n scales as 1/delta^2 — small lifts need enormous samples")

detect 0.5pp on 5% base: n = 31,217 per group
detect 1.0pp on 5% base: n = 8,143 per group
detect 2.0pp on 5% base: n = 2,198 per group

rule: n scales as 1/delta^2 — small lifts need enormous samples


**The follow-up:** *"the test needs 90k users, we only have 40k — options?"* — (1) accept lower power/MDE and say what you *can* detect, (2) run longer, (3) reduce variance: CUPED, stratification, or a more sensitive metric (continuous revenue vs binary conversion), (4) test a bigger change. Naming variance reduction is the senior move.

## Q3 — Peeking: Why You Can't Stop When p < 0.05

Checking the test continuously and stopping at the first significance turns α=0.05 into α≈0.25 — you're running many tests and reporting the winner. The executed demo below simulates exactly this. **The fixes, in order of correctness:** fix the horizon in advance; or use a sequential design (O'Brien-Fleming / alpha-spending, mSPRT); or Bayesian monitoring with thresholds stated in advance. Not "I'll be disciplined" — a procedure.

In [2]:
rng = np.random.default_rng(0)
n_max, n_peek = 20_000, 500          # check every 500 users

false_stops = 0
trials = 400
for _ in range(trials):
    # TRUE NULL: identical conversion in both arms
    a = rng.binomial(1, 0.10, n_max)
    b = rng.binomial(1, 0.10, n_max)
    stopped = False
    for i in range(n_peek, n_max + 1, n_peek):
        ca, cb = a[:i].sum(), b[:i].sum()
        p_pool = (ca + cb) / (2 * i)
        se = (p_pool * (1 - p_pool) * 2 / i) ** 0.5
        z = (cb / i - ca / i) / se
        if abs(z) > 1.96:
            stopped = True
            break
    false_stops += stopped

print(f"nominal alpha = 0.05, but peeking every {n_peek} users:")
print(f"false-positive rate = {false_stops/trials:.1%} across {trials} null simulations")
print("sequential looks multiply the real Type I error — this is why horizons are pre-registered")

nominal alpha = 0.05, but peeking every 500 users:
false-positive rate = 29.8% across 400 null simulations
sequential looks multiply the real Type I error — this is why horizons are pre-registered


## Q4 — CUPED: Variance Reduction in Twelve Lines

**CUPED** (Controlled-experiment Using Pre-Experiment Data): regress the metric on its pre-period value and test on the *residuals*. If pre-period data explains 30% of variance, you need ~30% fewer users — or equivalently, your confidence intervals tighten. The trick only works with a covariate measured **before randomization** (it's guaranteed unaffected by treatment).

In [3]:
rng = np.random.default_rng(1)
n = 5000
pre = rng.normal(100, 15, n)                    # pre-period metric per user
treat = rng.integers(0, 2, n)                   # random assignment
post = pre + treat * 1.2 + rng.normal(0, 12, n) # small true effect: +1.2

# naive comparison
d_naive = post[treat == 1].mean() - post[treat == 0].mean()
se_naive = np.sqrt(post[treat==1].var()/n + post[treat==0].var()/n)

# CUPED: remove the pre-period component
theta = np.cov(pre, post)[0, 1] / pre.var()
y_cuped = post - theta * (pre - pre.mean())
d_cuped = y_cuped[treat == 1].mean() - y_cuped[treat == 0].mean()
se_cuped = np.sqrt(y_cuped[treat==1].var()/n + y_cuped[treat==0].var()/n)

print(f"naive : effect {d_naive:+.3f} +/- {1.96*se_naive:.3f}  (true +1.2)")
print(f"CUPED : effect {d_cuped:+.3f} +/- {1.96*se_cuped:.3f}")
print(f"CI width reduced by {1 - se_cuped/se_naive:.0%} — pre-period data bought power for free")

naive : effect +0.868 +/- 0.760  (true +1.2)
CUPED : effect +1.340 +/- 0.468
CI width reduced by 38% — pre-period data bought power for free


## Q5 — Simpson's Reversal on Treatment Effects

*"Treatment looks better in every region but worse overall — possible?"* Yes, when allocation is confounded: the treatment group skews toward harder cases. This is *the* argument for randomization, and being able to produce the reversal on demand is a guaranteed memorable moment:

In [4]:
import pandas as pd

rows = []
# region A: 90% treated, low baseline; region B: 10% treated, high baseline
for region, base, p_treat in [("A", 0.30, 0.90), ("B", 0.55, 0.10)]:
    for _ in range(3000):
        t = rng.random() < p_treat
        outcome = rng.random() < base + (0.04 if t else 0)   # treatment adds +4pp EVERYWHERE
        rows.append((region, t, outcome))
df = pd.DataFrame(rows, columns=["region", "treated", "recovered"])

print("recovery rate by arm WITHIN each region (treatment truly helps both):")
print(df.groupby(["region", "treated"])["recovered"].mean().round(3).to_string())
overall = df.groupby("treated")["recovered"].mean()
print(f"\nPOOLED: control {overall.loc[False]:.3f} vs treated {overall.loc[True]:.3f} -> treatment looks WORSE overall")
print("allocation confounding flipped the sign — randomization is the only clean fix")

recovery rate by arm WITHIN each region (treatment truly helps both):
region  treated
A       False      0.301
        True       0.349
B       False      0.557
        True       0.595

POOLED: control 0.532 vs treated 0.374 -> treatment looks WORSE overall
allocation confounding flipped the sign — randomization is the only clean fix


## Q6 — Causal Inference: the Kitchen-Table Version

*“Why can't we just compare users who clicked vs didn't?”* — because clicking isn't random: selection bias. The causal toolkit to name-drop with one-liners:

- **Potential outcomes / RCT** — the gold standard: randomization makes groups exchangeable.
- **Difference-in-differences** — before/after between treatment and control regions; identifies the effect if *parallel trends* hold.
- **Instrumental variables** — a variable that affects treatment but not outcome except through treatment (e.g., server-side rollout %); recovers causal effect for compliers.
- **Regression discontinuity** — treatment assigned by a threshold (scholarship at a cutoff); compare just-below vs just-above.
- **Propensity matching / IPW** — model P(receiving treatment), reweight to a pseudo-population; only adjusts for *observed* confounders — say that limitation out loud.
- **Doubly robust / causal forests** — combine outcome model + propensity model; consistent if either is right. The modern estimator family behind "uplift modeling."

**Uplift modeling** — the ML-meets-caususion question: predict *individual treatment effect* (who converts *because* of the nudge), usually via the two-model trick or transformed-outcome trees. Metrics: Qini curve / uplift deciles — "treat the top decile only" is the money answer.

## Q7 — Quick-Fire Experimentation Round

| Question | The 15-second answer |
| --- | --- |
| One-tailed vs two-tailed? | Two-tailed unless the *only* acceptable direction was pre-committed; switching tails after seeing data is p-hacking |
| p-value 0.049 vs 0.051? | Identical in practice — report effect size + CI; the threshold is a convention, not a law of nature |
| Novelty effects? | Guardrail: holdback cohort, or analyze first-week vs later cohorts for trend |
| Interference between arms? | Marketplace/social networks: treatment spills to control (network effects); mitigations: cluster randomization, geo experiments, switchback designs |
| Multiple metrics/metrics? | Primary metric pre-registered; Bonferroni/FDR for secondary families; guardrails are one-sided tests |
| What's an A/A test? | Randomize identical experiences — validates the pipeline; nonzero "lift" in A/A means instrumentation bugs or peeking |
| Regression to the mean in experiments? | Extremes drift back; why "worst cohort improved most" isn't evidence the treatment worked on them |
| Ship/kill decision with p=0.07? | Look at CI width + direction consistency across segments + cost of being wrong — a narrow CI around a small positive effect often justifies shipping |